# DyTopo — Supply Chain Domain (SAP / SBN Doc Routing)

**Based on:** Lu et al. 2026 — *DyTopo: Dynamic Topology Routing for Multi-Agent Reasoning* (arXiv:2602.06039)  
**Applied to:** SBN AI Agentic Framework — Conversational Document Agent (SAP BTP / Joule / cXML)  
**Target runtime:** Google Colab G4 (NVIDIA RTX PRO 6000, 96 GB VRAM) or any GPU tier

## What this notebook does
Runs Algorithm 1 in **mock mode** (no LLM needed) on two supply chain scenarios:
- **Scenario A — Happy Path:** Invoice ingestion → validation → auto-repair → confirm → post
- **Scenario B — Routing Failure:** Clean invoice fails routing → InvestigatorAgent diagnoses root cause

## Agent mapping to SBN architecture
| DyTopo Agent | SBN Agent | Responsibility |
|---|---|---|
| `IngestAgent` | DocumentIngestionAgent | Extract fields, fingerprint, template lookup |
| `ValidatorAgent` | ValidationOrchestrator | Structural + domain business rule checks |
| `RepairAgent` | AutoRepairAgent | AUTO_ENRICHABLE fixes, NEEDS_USER escalation |
| `InvestigatorAgent` | InvestigationLoopAgent | Root cause ReAct loop for routing failures |

## Topology evolution expected
| Round | Stage | Expected density |
|---|---|---|
| 0 | Document ingestion + initial validation | High (~60-80%) |
| 1 | Repair loop — auto-enrich + ask user | Moderate (~25-45%) |
| 2 | Confirm + post | Low (<20%) |

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────────
!nvidia-smi
import subprocess
result = subprocess.run(
    ['python', '-c', 'import torch; print("CUDA:", torch.cuda.is_available()); '
     'print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")'],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q sentence-transformers httpx numpy matplotlib networkx pandas nest_asyncio
print('Done.')

In [ ]:
# ── Cell 3: Download engines from GitHub ───────────────────────────────────────
import urllib.request, os, sys

BASE_URL = 'https://raw.githubusercontent.com/51103220/agentic-dynamic-topology/refs/heads/main/api/'

for fname in ['dytopo_engine.py', 'supply_chain_engine.py']:
    path = f'/content/{fname}'
    try:
        urllib.request.urlretrieve(BASE_URL + fname, path)
        print(f'✓ Downloaded {fname} ({os.path.getsize(path):,} bytes)')
    except Exception as e:
        raise RuntimeError(
            f'Download failed for {fname}: {e}\n'
            'Manually upload both files to /content/ via the Files panel.'
        )

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import nest_asyncio
nest_asyncio.apply()

from dytopo_engine import build_adjacency, encode
from supply_chain_engine import (
    SUPPLY_CHAIN_ROLES,
    supply_chain_config,
    run_supply_chain,
)
print('✓ Both engines imported')

# Move encoder to CUDA if available
import torch
if torch.cuda.is_available():
    import dytopo_engine
    from sentence_transformers import SentenceTransformer
    dytopo_engine._encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    print('✓ Encoder on CUDA')

In [ ]:
# ── Cell 4 (optional): Configure LLM model based on available VRAM ─────────────
# Skip this cell if running mock_mode=True
# Run before switching mock_mode=False for real LLM inference
#
# Two serving backends supported:
#   Ollama  — pull quantised GGUF, easy setup, slightly lower quality
#   vLLM    — load full HF model, better quality, needs ~10 min first load

import torch

BACKEND = 'vllm'   # <── change to 'ollama' if you prefer Ollama

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vram_gb:.1f} GB')

    if BACKEND == 'vllm':
        # vLLM VRAM requirements (bf16 unless noted):
        #   Qwen/Qwen2.5-72B-Instruct   bf16  ~144 GB  (too large, need 2x GPU)
        #   Qwen/Qwen2.5-72B-Instruct   fp8   ~72 GB   ✓ fits 95 GB  ← best choice
        #   Qwen/Qwen2.5-32B-Instruct   bf16  ~64 GB   ✓ fits 95 GB
        #   Qwen/Qwen2.5-14B-Instruct   bf16  ~28 GB   ✓
        #   Qwen/Qwen2.5-7B-Instruct    bf16  ~14 GB   ✓
        #   Qwen/QwQ-32B                bf16  ~64 GB   ✓ (reasoning model)

        if vram_gb >= 90:
            LLM_MODEL    = 'Qwen/Qwen2.5-72B-Instruct'
            VLLM_EXTRA   = '--quantization fp8'        # fp8 keeps ~72 GB footprint
            note = 'Qwen2.5-72B fp8 — best quality for 95 GB single GPU'
        elif vram_gb >= 64:
            LLM_MODEL    = 'Qwen/Qwen2.5-72B-Instruct'
            VLLM_EXTRA   = '--quantization awq'
            note = 'Qwen2.5-72B AWQ — fits 64–89 GB'
        elif vram_gb >= 36:
            LLM_MODEL    = 'Qwen/Qwen2.5-32B-Instruct'
            VLLM_EXTRA   = ''
            note = 'Qwen2.5-32B bf16 — fits 36–63 GB'
        elif vram_gb >= 28:
            LLM_MODEL    = 'Qwen/Qwen2.5-14B-Instruct'
            VLLM_EXTRA   = ''
            note = 'Qwen2.5-14B bf16 — fits 28–35 GB'
        else:
            LLM_MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
            VLLM_EXTRA   = ''
            note = 'Qwen2.5-7B bf16 — T4 16 GB'

        LLM_BASE_URL = 'http://localhost:8000/v1'
        print(f'→ {LLM_MODEL}  ({note})')
        print()
        print('Start vLLM server with:')
        print(f'  !vllm serve {LLM_MODEL} {VLLM_EXTRA} --port 8000 &')
        print('  (wait ~60–300s for model to load before running inference cells)')

    else:  # ollama
        # Ollama GGUF quant tiers:
        if vram_gb >= 90:
            LLM_MODEL = 'qwen2.5:72b-instruct-q8_0'   # ~72 GB
            note = 'Qwen2.5-72B q8_0 — best Ollama quant for 95 GB'
        elif vram_gb >= 48:
            LLM_MODEL = 'qwen2.5:72b-instruct'         # q4_K_M ~45 GB
            note = 'Qwen2.5-72B q4_K_M — fits 48–89 GB'
        elif vram_gb >= 36:
            LLM_MODEL = 'qwen2.5:32b-instruct'
            note = 'Qwen2.5-32B — fits 36–47 GB'
        else:
            LLM_MODEL = 'qwen2.5:7b-instruct'
            note = 'Qwen2.5-7B — small GPU'

        LLM_BASE_URL = 'http://localhost:11434/v1'
        print(f'→ {LLM_MODEL}  ({note})')
        print()
        print('Pull model with:')
        print(f'  !ollama pull {LLM_MODEL}')

else:
    LLM_MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
    LLM_BASE_URL = 'http://localhost:8000/v1'
    VLLM_EXTRA   = ''
    print(f'No GPU — CPU fallback: {LLM_MODEL}')

print()
print('Pass to supply_chain_config() when switching to real LLM:')
print(f'  mock_mode=False, llm_model="{LLM_MODEL}", llm_base_url="{LLM_BASE_URL}"')

In [ ]:
# ── Cell 4: Run Scenario A — Happy Path ────────────────────────────────────────
import asyncio

INVOICE_DOC = (
    'Invoice INV-2026-001 from ABC Corp to buyer BNO-456001. '
    'PO reference: PO-98821. Date: 2026-10-01. Currency: USD. '
    'Line items: 5x Widget-A at $320.00 each. '
    'GRN remaining quantity: 8 units. '
    'Submit to Doc Routing Egress API as cXML.'
)

cfg = supply_chain_config(tau=0.35, max_rounds=5, mock_mode=True)

async def collect_happy_path():
    results = []
    async for rr in run_supply_chain(INVOICE_DOC, cfg, scenario='happy_path'):
        results.append(rr)
        A = rr.adjacency
        N = len(A)
        edges = sum(A[j][i] for j in range(N) for i in range(N) if j != i)
        density = edges / (N * (N - 1))
        print(f'  Round {rr.round + 1} [{rr.goal[:55]}...]')
        print(f'    edges={edges}  density={density:.0%}  complete={rr.is_complete}')
    return results

print('=== Scenario A: Happy Path (Invoice INV-2026-001) ===')
happy_rounds = asyncio.run(collect_happy_path())
print(f'\n✓ {len(happy_rounds)} rounds completed')
N = len(SUPPLY_CHAIN_ROLES)
print(f'Agents: {SUPPLY_CHAIN_ROLES}')

In [ ]:
# ── Cell 5: Run Scenario B — Routing Failure ───────────────────────────────────
FAILED_INVOICE_DOC = (
    'Invoice INV-78432 from XYZ Ltd to buyer BNO-456001. '
    'Document validated and clean. '
    'Submitted to Doc Routing Egress API — routing failure detected. '
    'Investigate why INV-78432 failed routing.'
)

cfg_failure = supply_chain_config(tau=0.35, max_rounds=5, mock_mode=True)

async def collect_routing_failure():
    results = []
    async for rr in run_supply_chain(FAILED_INVOICE_DOC, cfg_failure, scenario='routing_failure'):
        results.append(rr)
        A = rr.adjacency
        N = len(A)
        edges = sum(A[j][i] for j in range(N) for i in range(N) if j != i)
        density = edges / (N * (N - 1))
        print(f'  Round {rr.round + 1} — edges={edges}  density={density:.0%}  complete={rr.is_complete}')
    return results

print('=== Scenario B: Routing Failure (Invoice INV-78432) ===')
failure_rounds = asyncio.run(collect_routing_failure())
print(f'\n✓ {len(failure_rounds)} rounds completed')

In [ ]:
# ── Cell 6: Adjacency matrix heatmaps — Happy Path ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'text.color': 'white', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white'})

BG, PANEL = '#0a0e1a', '#0f1828'
PHASES = ['Ingestion & Initial Validation', 'Repair Loop & Enrichment', 'Confirm & Post']
ROLES_SHORT = ['Ingest', 'Validator', 'Repair', 'Investigator']

def plot_heatmaps(rounds, title_prefix, save_path):
    n_rounds = len(rounds)
    fig, axes = plt.subplots(1, n_rounds, figsize=(4.8 * n_rounds, 5))
    fig.patch.set_facecolor(BG)
    if n_rounds == 1: axes = [axes]

    for ax, rr in zip(axes, rounds):
        A = np.array(rr.adjacency)
        edges   = int(A.sum())
        density = edges / (N * (N - 1))
        phase   = PHASES[min(rr.round, len(PHASES) - 1)]

        ax.set_facecolor(PANEL)
        ax.imshow(A, cmap='Blues', vmin=0, vmax=1, aspect='equal')
        ax.set_xticks(range(N)); ax.set_xticklabels(ROLES_SHORT, rotation=40, ha='right', fontsize=8)
        ax.set_yticks(range(N)); ax.set_yticklabels(ROLES_SHORT, fontsize=8)
        ax.set_title(f'Round {rr.round + 1} — {phase}\n{edges} edges · {density:.0%} density',
                     fontsize=8.5, pad=8)
        ax.set_xlabel('Source (provides key)', fontsize=8)
        ax.set_ylabel('Destination (needs query)', fontsize=8)

        for i in range(N):
            for j in range(N):
                ax.text(j, i, str(A[i, j]), ha='center', va='center', fontsize=12,
                        color='white' if A[i, j] else '#334155', fontweight='bold')

    fig.suptitle(
        f'DyTopo Supply Chain — {title_prefix}\n'
        'A[j→i]=1 when key_j semantically matches query_i above τ=0.35',
        fontsize=11, fontweight='bold', color='white', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f'Saved: {save_path}')

plot_heatmaps(happy_rounds, 'Happy Path (INV-2026-001)',
              '/content/sc_heatmap_happy.png')
plot_heatmaps(failure_rounds, 'Routing Failure (INV-78432)',
              '/content/sc_heatmap_failure.png')

In [ ]:
# ── Cell 7: Directed topology graphs (NetworkX) ────────────────────────────────
import networkx as nx

ROLE_COLORS = {
    'IngestAgent':       '#38bdf8',   # sky blue
    'ValidatorAgent':    '#fbbf24',   # amber
    'RepairAgent':       '#4ade80',   # green
    'InvestigatorAgent': '#f87171',   # red
}

G_ref = nx.DiGraph()
G_ref.add_nodes_from(range(N))
pos = nx.circular_layout(G_ref)

def plot_topology_graphs(rounds, title_prefix, save_path):
    n_rounds = len(rounds)
    fig, axes = plt.subplots(1, n_rounds, figsize=(5.5 * n_rounds, 5.5))
    fig.patch.set_facecolor(BG)
    if n_rounds == 1: axes = [axes]

    for ax, rr in zip(axes, rounds):
        A = np.array(rr.adjacency)
        G = nx.DiGraph()
        G.add_nodes_from(range(N))
        for i in range(N):
            for j in range(N):
                if A[i, j]: G.add_edge(i, j)

        node_colors = [ROLE_COLORS.get(SUPPLY_CHAIN_ROLES[i], '#94a3b8') for i in range(N)]
        node_sizes  = [1000 + 300 * G.in_degree(i) for i in range(N)]
        labels      = {i: ROLES_SHORT[i] for i in range(N)}

        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                               node_size=node_sizes, alpha=0.92)
        nx.draw_networkx_labels(G, pos, labels=labels, ax=ax,
                                font_size=8, font_color='white', font_weight='bold')
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#64748b', arrows=True,
                               arrowstyle='-|>', arrowsize=20,
                               connectionstyle='arc3,rad=0.15', width=1.8)

        edges   = int(A.sum())
        density = edges / (N * (N - 1))
        phase   = PHASES[min(rr.round, len(PHASES) - 1)]
        ax.set_title(f'Round {rr.round + 1} — {phase}\n{edges} edges · {density:.0%} density',
                     fontsize=9, color='white', pad=10)
        ax.axis('off'); ax.set_facecolor(BG)

    # Legend
    from matplotlib.patches import Patch
    legend = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
    axes[-1].legend(handles=legend, loc='lower right',
                    facecolor='#1a2235', edgecolor='#1e293b',
                    labelcolor='white', fontsize=8)

    fig.suptitle(f'DyTopo Supply Chain Topology — {title_prefix}\nReproduces dense→sparse pattern from paper Figure 4',
                 fontsize=11, fontweight='bold', color='white', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f'Saved: {save_path}')

plot_topology_graphs(happy_rounds, 'Happy Path', '/content/sc_topology_happy.png')
plot_topology_graphs(failure_rounds, 'Routing Failure', '/content/sc_topology_failure.png')

In [ ]:
# ── Cell 8: Edge density evolution — Happy Path vs Routing Failure ─────────────
def get_densities(rounds):
    return [
        (rr.round + 1, np.array(rr.adjacency).sum() / (N * (N - 1)))
        for rr in rounds
    ]

happy_dens   = get_densities(happy_rounds)
failure_dens = get_densities(failure_rounds)

PHASE_COLORS = ['#38bdf8', '#fbbf24', '#4ade80']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('DyTopo Supply Chain — Edge Density Evolution\nDense ingestion → sparse finalization',
             fontsize=12, color='white', fontweight='bold', y=1.02)

for ax, dens, title, color in [
    (axes[0], happy_dens,   'Scenario A: Happy Path',       '#4ade80'),
    (axes[1], failure_dens, 'Scenario B: Routing Failure',  '#f87171'),
]:
    ax.set_facecolor(PANEL)
    xs = [x for x, _ in dens]
    ys = [y for _, y in dens]

    ax.plot(xs, ys, 'o-', color=color, linewidth=2.5, markersize=10, zorder=3)
    ax.fill_between(xs, ys, alpha=0.1, color=color)

    for i, (x, y) in enumerate(zip(xs, ys)):
        phase = PHASES[min(i, len(PHASES) - 1)]
        col   = PHASE_COLORS[min(i, len(PHASE_COLORS) - 1)]
        ax.annotate(f'{y:.0%}\n{phase.split(" & ")[0]}',
                    xy=(x, y), xytext=(0, 18),
                    textcoords='offset points', ha='center', fontsize=8,
                    color=col, fontweight='bold')

    ax.set_xlabel('Communication Round', fontsize=11, color='white')
    ax.set_ylabel('Edge Density', fontsize=11, color='white')
    ax.set_title(title, fontsize=10, color='white')
    ax.set_ylim(0, 1.3)
    ax.set_xticks(xs)
    ax.set_xticklabels([f'Round {x}' for x in xs])
    ax.grid(axis='y', alpha=0.2)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#1e293b')

plt.tight_layout()
plt.savefig('/content/sc_density_evolution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: /content/sc_density_evolution.png')

In [ ]:
# ── Cell 9: Q/K output table + in-degree analysis ──────────────────────────────
import pandas as pd

def print_qk_table(rounds, scenario_name):
    rows = []
    for rr in rounds:
        A = np.array(rr.adjacency)
        for a in rr.agents:
            in_deg = int(A[:, a.id].sum())
            rows.append({
                'Round':          rr.round + 1,
                'Role':           a.role,
                'In-degree':      in_deg,
                'In-neighbors':   [SUPPLY_CHAIN_ROLES[j] for j in a.in_neighbors],
                'Q-vector':       a.q_vector[:65] + ('...' if len(a.q_vector) > 65 else ''),
                'K-vector':       a.k_vector[:65] + ('...' if len(a.k_vector) > 65 else ''),
                'Output (trunc)': a.public_content[:75] + ('...' if len(a.public_content) > 75 else ''),
            })

    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 80)
    pd.set_option('display.width', 180)
    print(f'\n{"="*100}')
    print(f'  Q/K Output Table — {scenario_name}')
    print(f'{"="*100}')
    print(df.to_string(index=False))

    # Final answer
    for rr in rounds:
        if rr.final_answer:
            print(f'\n══ Final Answer (Round {rr.round + 1}) ══')
            print(rr.final_answer)

print_qk_table(happy_rounds,   'Scenario A — Happy Path')
print_qk_table(failure_rounds, 'Scenario B — Routing Failure')

In [ ]:
# ── Cell 10: In-degree evolution per agent ─────────────────────────────────────
# Shows which agent is the communication hub per round
# InvestigatorAgent should drop to 0 in-degree on happy path

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('In-degree per Agent per Round\nIn-degree=0 means agent is isolated (receives no messages)',
             fontsize=11, color='white', fontweight='bold')

for ax, rounds, title in [
    (axes[0], happy_rounds,   'Scenario A: Happy Path'),
    (axes[1], failure_rounds, 'Scenario B: Routing Failure'),
]:
    ax.set_facecolor(PANEL)
    round_nums = [rr.round + 1 for rr in rounds]

    for agent_idx, (role, color) in enumerate(ROLE_COLORS.items()):
        in_degrees = []
        for rr in rounds:
            A = np.array(rr.adjacency)
            in_degrees.append(int(A[:, agent_idx].sum()))
        ax.plot(round_nums, in_degrees, 'o-', color=color, linewidth=2.2,
                markersize=8, label=ROLES_SHORT[agent_idx])

    ax.set_xlabel('Round', color='white', fontsize=11)
    ax.set_ylabel('In-degree (messages received)', color='white', fontsize=11)
    ax.set_title(title, color='white', fontsize=10)
    ax.set_xticks(round_nums)
    ax.set_xticklabels([f'Round {x}' for x in round_nums])
    ax.set_ylim(-0.2, N)
    ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=9)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#1e293b')
    ax.grid(axis='y', alpha=0.2)
    ax.axhline(0, color='#475569', linewidth=1, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('/content/sc_indegree_evolution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: /content/sc_indegree_evolution.png')
print()
print('Key insight: InvestigatorAgent in-degree stays at 0 on happy path')
print('→ zero tokens consumed by investigator when routing succeeds')
print('→ DyTopo naturally routes it out of the graph without hardcoding')

In [ ]:
# ── Cell 11: τ sweep for supply chain ─────────────────────────────────────────
# Find the optimal tau for supply chain domain (analogous to paper Table 3)
# τ=0.3 is optimal for code, τ=0.4 for math — what is it for supply chain?

from supply_chain_engine import DyTopoConfig

TAU_VALUES = [round(t, 2) for t in np.arange(0.10, 0.80, 0.05)]

async def run_tau_sweep_sc(tau: float):
    cfg = DyTopoConfig(
        task_type='supply_chain', n_agents=4, tau=tau,
        max_rounds=3, mock_mode=True,
    )
    round_densities = []
    async for rr in run_supply_chain(INVOICE_DOC, cfg, scenario='happy_path'):
        A = np.array(rr.adjacency)
        density = A.sum() / (N * (N - 1))
        round_densities.append((rr.round, float(density)))
    return round_densities

print(f'Sweeping {len(TAU_VALUES)} tau values for supply chain domain...')
tau_results_sc = {}
for tau in TAU_VALUES:
    rd = asyncio.run(run_tau_sweep_sc(tau))
    tau_results_sc[tau] = rd
    avg = np.mean([d for _, d in rd])
    bar = '█' * int(avg * 20)
    print(f'  τ={tau:.2f}  avg_density={avg:5.1%}  {bar}')

print(f'\n✓ Supply chain τ sweep complete')

In [ ]:
# ── Cell 12: τ sweep chart — supply chain vs code vs math ─────────────────────
taus = np.array(sorted(tau_results_sc.keys()))
avg_dens_sc = np.array([np.mean([d for _, d in tau_results_sc[t]]) for t in taus])

# Paper optimal points for reference
TAU_PAPER   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
APPS_COMP   = [43.61, 44.84, 49.81, 47.35, 44.13, 40.86, 40.47, 37.59, 34.51]
OMNI_MATH   = [42.86, 47.14, 50.00, 52.86, 48.57, 42.86, 41.43, 40.00, 38.58]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.patch.set_facecolor(BG)
fig.suptitle('τ Analysis — Supply Chain Domain\nComparing structural density to paper accuracy curves',
             fontsize=11, color='white', fontweight='bold')

# Left: supply chain density vs tau with regime bands
ax = axes[0]
ax.set_facecolor(PANEL)

def regime_color(d):
    if d > 0.70: return '#f87171'
    elif d < 0.10: return '#fbbf24'
    return '#4ade80'

bar_colors = [regime_color(d) for d in avg_dens_sc]
ax.bar(taus, avg_dens_sc, width=0.04, color=bar_colors, alpha=0.85, zorder=3)
ax.plot(taus, avg_dens_sc, 'o-', color='white', linewidth=1.4, markersize=4, alpha=0.6, zorder=4)

ax.axhspan(0.10, 0.70, alpha=0.07, color='#4ade80', label='Optimal band (10–70%)')
ax.axhspan(0.70, 1.00, alpha=0.07, color='#f87171')
ax.axhspan(0.00, 0.10, alpha=0.07, color='#fbbf24')
ax.axvline(x=0.35, color='#38bdf8', linestyle='--', linewidth=1.8, alpha=0.8,
           label='τ=0.35 (supply chain default)')

ax.text(0.11, 0.82, 'OVER-DENSE', color='#f87171', fontsize=8)
ax.text(0.11, 0.38, 'OPTIMAL RANGE', color='#4ade80', fontsize=8)
ax.text(0.11, 0.04, 'SPARSE', color='#fbbf24', fontsize=8)

ax.set_xlabel('Similarity Threshold τ', fontsize=10, color='white')
ax.set_ylabel('Avg Edge Density', fontsize=10, color='white')
ax.set_title('Supply Chain — Avg Edge Density vs τ', fontsize=10, color='white')
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b')
ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=8)
ax.grid(axis='y', alpha=0.35, zorder=0); ax.set_ylim(0, 1.05)

# Right: paper accuracy curves for comparison
ax2 = axes[1]
ax2.set_facecolor(PANEL)
ax2.plot(TAU_PAPER, APPS_COMP, 'o-', color='#38bdf8', linewidth=2.2, markersize=7,
         label='APPS-Competition (τ*=0.3, code)')
ax2.plot(TAU_PAPER, OMNI_MATH, 's-', color='#fbbf24', linewidth=2.2, markersize=7,
         label='Omni-Math (τ*=0.4, math)')
ax2.axvline(x=0.35, color='#4ade80', linestyle='--', linewidth=1.8,
            label='τ=0.35 (supply chain default)')
ax2.set_xlabel('Similarity Threshold τ', fontsize=10, color='white')
ax2.set_ylabel('Accuracy (%)', fontsize=10, color='white')
ax2.set_title('Paper Table 3 — Code and Math Optima (reference)', fontsize=10, color='white')
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#1e293b')
ax2.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=8)
ax2.grid(axis='y', alpha=0.35); ax2.set_ylim(30, 57)

plt.tight_layout()
plt.savefig('/content/sc_tau_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: /content/sc_tau_analysis.png')

In [ ]:
# ── Cell 13: Summary — SBN architecture mapping ────────────────────────────────
import pandas as pd

print('=' * 90)
print('  DyTopo Supply Chain — Summary')
print('  Domain: SAP SBN Conversational Document Agent (cXML / B2B / Joule)')
print('=' * 90)

print()
print('AGENT MAPPING')
mapping = pd.DataFrame([
    {'DyTopo Role': 'IngestAgent',       'SBN Agent': 'DocumentIngestionAgent',
     'Responsibility': 'Extract fields, fingerprint, template registry lookup'},
    {'DyTopo Role': 'ValidatorAgent',    'SBN Agent': 'ValidationOrchestrator',
     'Responsibility': 'Structural + domain business rule checks, violation classification'},
    {'DyTopo Role': 'RepairAgent',       'SBN Agent': 'AutoRepairAgent',
     'Responsibility': 'Silent AUTO_ENRICHABLE fixes, NEEDS_USER escalation (1 Q per turn)'},
    {'DyTopo Role': 'InvestigatorAgent', 'SBN Agent': 'InvestigationLoopAgent',
     'Responsibility': 'Root cause ReAct loop (FlowLog, Kafka, RoutingDecision, OrgConfig)'},
])
print(mapping.to_string(index=False))

print()
print('TOPOLOGY EVOLUTION')
topology = pd.DataFrame([
    {'Round': 1, 'Stage': 'Ingestion & Validation', 'Expected Density': 'HIGH (~60-80%)',
     'Why': 'All agents share document context broadly'},
    {'Round': 2, 'Stage': 'Repair & Enrich',        'Expected Density': 'MODERATE (~25-45%)',
     'Why': 'Only repair loop agents active (Validator ↔ Repair ↔ Ingest)'},
    {'Round': 3, 'Stage': 'Confirm & Post',          'Expected Density': 'LOW (<20%)',
     'Why': 'All agents emit "I need nothing further" → edges drop to zero'},
])
print(topology.to_string(index=False))

print()
print('KEY INSIGHT: InvestigatorAgent on happy path')
for rr in happy_rounds:
    A = np.array(rr.adjacency)
    inv_indeg = int(A[:, 3].sum())   # InvestigatorAgent is index 3
    print(f'  Round {rr.round + 1}: InvestigatorAgent in-degree = {inv_indeg}', end='')
    print(' ← ISOLATED (zero tokens consumed)' if inv_indeg == 0 else ' ← ACTIVE')

print()
print('→ DyTopo routes InvestigatorAgent out of the graph automatically')
print('  when routing succeeds — no hardcoded if/else needed.')
print('  Matches SBN design: InvestigationLoopAgent only activates on routing failure.')

print()
print('Saved plots:')
for f in ['sc_heatmap_happy.png', 'sc_heatmap_failure.png',
          'sc_topology_happy.png', 'sc_topology_failure.png',
          'sc_density_evolution.png', 'sc_indegree_evolution.png',
          'sc_tau_analysis.png']:
    print(f'  /content/{f}')